# Practice 3 - Exercise 2: Dataset & Preprocessing
**Thành viên phụ trách:** Văn Sơn
**Vai trò:** DATASET & PREPROCESSING
**Phạm vi:** Đây là phần đầu của Exercise 2 (Finetuning a Pretrained Model for Binary Text Classification)

Notebook này thực hiện:
1. Load một dataset đơn giản cho bài toán **binary text classification**
2. Kiểm tra (inspect) dataset
3. Tiền xử lý (cleaning + tokenization)
4. Chia train/test (nếu cần)
5. Bàn giao: Processed Dataset + Tokenizer preprocessing code + Dataset information


## Bước 0 – Cài đặt thư viện cần thiết

In [6]:
!pip install -q transformers datasets evaluate

'c:\Users\LENOVO\Downloads\UTH-Deep-Learning-nhom2\.venv\Scripts\pip.exe' was blocked by your organization's Device Guard policy.
Contact your support person for more info.


## Bước 1 – Load dataset

Sử dụng dataset **IMDB** (`imdb`) từ Hugging Face Hub — đây là dataset kinh điển cho
bài toán **binary text classification** (sentiment: positive / negative), cấu trúc gồm 2 cột:

```
Dataset
├── text   (str)
└── label  (0 = negative, 1 = positive)
```


In [7]:
from datasets import load_dataset

# Load dataset IMDB - binary text classification (positive / negative)
raw_dataset = load_dataset("stanfordnlp/imdb")

print(raw_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


> Ghi chú: Nếu nhóm muốn dùng dataset khác (ví dụ SST-2 trong bộ GLUE), chỉ cần đổi
> `load_dataset("imdb")` thành `load_dataset("glue", "sst2")` và đổi tên cột tương ứng
> ở các bước bên dưới (`sentence`/`label` thay vì `text`/`label`).


## Bước 2 – Kiểm tra dataset (Dataset Inspection)

Phân tích các thông tin:
- Số lượng dữ liệu (train/test)
- Tên các cột
- Xem thử text
- Xem thử label
- Số lượng class
- Kiểm tra missing values


In [8]:
# Số lượng mẫu ở mỗi split
for split in raw_dataset.keys():
    print(f"{split}: {len(raw_dataset[split])} samples")

train: 25000 samples
test: 25000 samples
unsupervised: 50000 samples


In [9]:
# Tên các cột (features)
print("Column names:", raw_dataset["train"].column_names)
print("Features:", raw_dataset["train"].features)

Column names: ['text', 'label']
Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [10]:
# Xem thử một vài mẫu (text + label)
for i in range(3):
    sample = raw_dataset["train"][i]
    print(f"--- Sample {i} ---")
    print("Label:", sample["label"])
    print("Text :", sample["text"][:300], "...\n")

--- Sample 0 ---
Label: 0
Text : I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...

--- Sample 1 ---
Label: 0
Text : "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity ...

--- Sample 2 ---
Label: 0
Text : If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer com

In [11]:
# Số lượng class và phân bố label (kiểm tra dataset có cân bằng không)
import pandas as pd

train_labels = raw_dataset["train"]["label"]
label_counts = pd.Series(train_labels).value_counts().sort_index()
print("Số lượng class:", len(label_counts))
print("Phân bố label (train):")
print(label_counts)

Số lượng class: 2
Phân bố label (train):
0    12500
1    12500
Name: count, dtype: int64


In [12]:
# Kiểm tra missing values (text rỗng hoặc None)
def count_missing(dataset_split):
    n_missing = sum(1 for t in dataset_split["text"] if t is None or str(t).strip() == "")
    return n_missing

for split in raw_dataset.keys():
    print(f"Missing values in '{split}':", count_missing(raw_dataset[split]))

Missing values in 'train': 0
Missing values in 'test': 0
Missing values in 'unsupervised': 0


## Bước 3 – Preprocessing

Pipeline tiền xử lý:

```
Raw Dataset
      ↓
Cleaning / Formatting
      ↓
Tokenizer
      ↓
Processed Dataset
```

### 3.1 Cleaning / Formatting
Dataset IMDB gốc chứa các thẻ HTML (ví dụ `<br />`) do được crawl từ web, cần loại bỏ
trước khi tokenize.


In [13]:
import re

def clean_text(example):
    text = example["text"]
    text = re.sub(r"<br\s*/?>", " ", text)      # xóa thẻ HTML <br />
    text = re.sub(r"\s+", " ", text).strip()     # gộp khoảng trắng thừa
    example["text"] = text
    return example

cleaned_dataset = raw_dataset.map(clean_text)

# Kiểm tra kết quả cleaning
print(cleaned_dataset["train"][0]["text"][:300])

Map: 100%|██████████| 50000/50000 [00:06<00:00, 7999.82 examples/s]

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h


### 3.2 Tokenizer
Dùng tokenizer tương ứng với pretrained model sẽ được finetune ở bước sau
(ví dụ `distilbert-base-uncased`).


In [14]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # có thể đổi theo model nhóm chọn để finetune
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

tokenized_dataset = cleaned_dataset.map(tokenize_function, batched=True)

# Xóa cột text gốc, đổi tên label -> labels (đúng convention của Trainer)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

c:\Users\LENOVO\Downloads\UTH-Deep-Learning-nhom2\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 50000/50000 [00:14<00:00, 3460.73 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [15]:
# Kiểm tra 1 mẫu sau khi tokenize
sample = tokenized_dataset["train"][0]
print("Keys:", sample.keys())
print("input_ids (10 token đầu):", sample["input_ids"][:10])
print("labels:", sample["labels"])

Keys: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])
input_ids (10 token đầu): tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026])
labels: tensor(0)


## Bước 4 – Chia dataset (train/test/validation)

Dataset IMDB đã có sẵn 2 split `train` và `test`. Để phục vụ việc theo dõi quá trình
finetune (early stopping, chọn checkpoint tốt nhất), tách thêm 10% từ `train` làm
**validation set**.


In [16]:
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)

final_dataset = {
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": tokenized_dataset["test"],
}

for name, ds in final_dataset.items():
    print(f"{name}: {len(ds)} samples")

train: 22500 samples
validation: 2500 samples
test: 25000 samples


## Đầu ra (Deliverables)

- **Processed Dataset**: `final_dataset` (train / validation / test), đã clean + tokenize,
  format `torch`, sẵn sàng đưa vào `Trainer`.
- **Tokenizer preprocessing code**: hàm `clean_text` + `tokenize_function` ở Bước 3.
- **Dataset information**: tổng hợp bên dưới.


In [17]:
from datasets import DatasetDict

final_dataset = DatasetDict(final_dataset)

# Lưu dataset đã xử lý ra đĩa để Exercise 2 phần sau (model + training) load lại
final_dataset.save_to_disk("processed_imdb_dataset")

# Lưu tokenizer cùng bộ để đảm bảo đồng bộ khi finetune
tokenizer.save_pretrained("processed_imdb_dataset/tokenizer")

print("Đã lưu dataset đã xử lý tại: processed_imdb_dataset/")

Saving the dataset (1/1 shards): 100%|██████████| 25000/25000 [00:00<00:00, 122024.07 examples/s]

Đã lưu dataset đã xử lý tại: processed_imdb_dataset/


In [18]:
# Tổng hợp Dataset information
dataset_info = {
    "dataset_name": "imdb",
    "task": "binary text classification (sentiment)",
    "num_classes": 2,
    "label_map": {0: "negative", 1: "positive"},
    "tokenizer": MODEL_NAME,
    "max_length": 256,
    "splits": {name: len(ds) for name, ds in final_dataset.items()},
}

for k, v in dataset_info.items():
    print(f"{k}: {v}")

dataset_name: imdb
task: binary text classification (sentiment)
num_classes: 2
label_map: {0: 'negative', 1: 'positive'}
tokenizer: distilbert-base-uncased
max_length: 256
splits: {'train': 22500, 'validation': 2500, 'test': 25000}


## Checklist nghiệm thu – Văn Sơn

- [x] Binary dataset (IMDB – 2 class: positive/negative)
- [x] Dataset inspection (số lượng, cột, class, missing values)
- [x] Text/label (xem thử mẫu text và label)
- [x] Preprocessing (cleaning HTML tags, format text)
- [x] Tokenization (dùng `AutoTokenizer` của `distilbert-base-uncased`)
- [x] Processed dataset (đã lưu tại `processed_imdb_dataset/`, sẵn sàng cho bước Model + Trainer)
